# Flux Bubble Explorer (v0.1.4)

Interactive exploration: toroidal/3D wall, warp shift, 3D surface + flow quiver.

**Modules:** `hfb.bubble`, `hfb.defects`, `hfb.utils.viz`

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

from hfb.bubble.stability import bubble_stability_metrics
from hfb.bubble.warp_conduit import flux_bubble_metric
from hfb.optics.raytrace import trace_rays_conformal
from hfb.utils.grid import cartesian_grid
from hfb.utils.viz import plot_flux_bubble_3d

NX, EXTENT = 96, 4.0
x, y = cartesian_grid(NX, NX, extent=EXTENT)
dx = float(x[0, 1] - x[0, 0])

In [ ]:
def explore_bubble(
    radius=1.0,
    wall_width=0.25,
    minor_radius=0.35,
    circulation=0.35,
    hopf_index=1,
    use_3d_torus=True,
    show_3d=True,
):
    metric = flux_bubble_metric(
        x, y,
        bubble_radius=radius,
        wall_width=wall_width,
        circulation=circulation,
        dx=dx,
        defect_profile="toroidal_bubble_wall",
        major_radius=radius,
        minor_radius=minor_radius,
        use_3d_torus=use_3d_torus,
        hopf_index=hopf_index,
    )
    report = bubble_stability_metrics(
        metric, dx, x=x, y=y,
        major_radius=radius, hopf_index=hopf_index,
        use_3d_torus=use_3d_torus,
    )

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, key, title in zip(axes, ["omega", "shift", "defect_density"],
                              ["Conformal Ω", "Shift", "Defect λ"]):
        im = ax.imshow(metric[key], origin="lower", extent=[-EXTENT, EXTENT]*2)
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(
        f"stable={report.stable_proxy} | max|R|={report.max_ricci:.2f} | "
        f"Φ_R={report.curvature_flux:.3f} | link={report.linking_proxy:.1f}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

    if show_3d:
        fig3d = plot_flux_bubble_3d(metric["omega"], metric["vx"], metric["vy"], dx=dx, extent=EXTENT)
        plt.show()

    fig2, ax2 = plt.subplots(figsize=(5, 5))
    for angle in np.linspace(-0.5, 0.5, 7):
        rx, ry = trace_rays_conformal(metric["omega"], -EXTENT*0.75, angle*1.5, angle, dx)
        ax2.plot(rx, ry, lw=0.8)
    ax2.set_xlim(-EXTENT, EXTENT)
    ax2.set_ylim(-EXTENT, EXTENT)
    ax2.set_aspect("equal")
    ax2.set_title("Conformal geodesics")
    plt.show()

interact(
    explore_bubble,
    radius=FloatSlider(0.6, 2.0, 0.05, value=1.0, description="radius"),
    wall_width=FloatSlider(0.1, 0.6, 0.02, value=0.25, description="wall_width"),
    minor_radius=FloatSlider(0.1, 0.8, 0.02, value=0.35, description="minor_R"),
    circulation=FloatSlider(0.0, 0.8, 0.05, value=0.35, description="circulation"),
    hopf_index=IntSlider(1, 4, 1, value=1, description="hopf_index"),
    use_3d_torus=Checkbox(value=True, description="use_3d_torus"),
    show_3d=Checkbox(value=True, description="show_3d"),
)